## Agent IA Conversationnel Juridique : Cas d’Usage Simplifié

## Contexte du Cas d’Usage

Un cabinet d’avocats souhaite automatiser la première interaction avec des clients potentiels concernant des questions courantes de droit du travail français. L’agent IA juridique permet de qualifier les demandes, fournir des informations générales et orienter vers le bon spécialiste.

## Exemple de Conversation

**Client** : “Bonjour, mon employeur vient de me licencier après 5 ans d’ancienneté. Ai-je droit à des indemnités ?”

**Agent IA** : “Bonjour, je comprends votre situation concernant votre licenciement. Pour vous fournir des informations générales, j’ai besoin de quelques précisions :

1. S’agit-il d’un licenciement pour motif personnel ou économique ?
2. Avez-vous reçu une lettre de licenciement ?
3. Quel est votre salaire brut mensuel moyen ?

⚠️ *Rappel : Les informations que je fournis sont à titre informatif général. Seul un avocat peut analyser votre situation spécifique et vous conseiller juridiquement.*”

**Client** : “C’est un licenciement économique. Mon salaire brut est de 2 800 € par mois.”

**Agent IA** : “Merci pour ces précisions. En cas de licenciement économique avec 5 ans d’ancienneté, voici les indemnités légales minimales auxquelles vous pourriez prétendre :

**Indemnité légale de licenciement** :
....
**Je vous recommande vivement de consulter un avocat spécialisé** en droit du travail pour :
....

**Client** : “Oui, je souhaite prendre rendez-vous.”

**Agent IA** : “Parfait. Je vous propose les créneaux suivants avec Maître Dupont, spécialisée en droit du travail :
....


In [ ]:
from typing import Dict, List
import anthropic

class AgentJuridiqueIA:
    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.historique_conversation = []
        
    def construire_prompt_systeme(self) -> str:
        return """Tu es un agent IA spécialisé en droit du travail français.

RÈGLES IMPÉRATIVES :
- Tu fournis uniquement des informations juridiques GÉNÉRALES
- Tu ne donnes JAMAIS de conseil juridique personnalisé
- Tu rappelles systématiquement que seul un avocat peut analyser une situation spécifique
- Tu inclus toujours un disclaimer sur la nature informative de tes réponses
- Tu recommandes la consultation d'un avocat pour tout cas concret

OUTILS DISPONIBLES :
- calculer_indemnites_licenciement : calcule les indemnités légales minimales
- rechercher_jurisprudence : recherche dans la base de jurisprudence
- prendre_rendez_vous : propose des créneaux avec un avocat

CONNAISSANCES :
- Code du travail français à jour
- Conventions collectives principales
- Jurisprudence récente

Ton objectif : qualifier la demande, informer généralement, orienter vers un avocat."""

    def calculer_indemnites_licenciement(
        self, 
        anciennete_annees: float, 
        salaire_brut_mensuel: float
    ) -> Dict:
        """Calcul des indemnités légales minimales de licenciement"""
        
        # Indemnité légale de licenciement
        if anciennete_annees < 10:
            indemnite_legale = (anciennete_annees * 0.25) * salaire_brut_mensuel
        else:
            indemnite_legale = (10 * 0.25 + (anciennete_annees - 10) * 0.33) * salaire_brut_mensuel
        
        # Préavis (simplifié - dépend de la catégorie professionnelle)
        if anciennete_annees < 2:
            preavis_mois = 1
        else:
            preavis_mois = 2
        
        indemnite_preavis = preavis_mois * salaire_brut_mensuel
        
        return {
            "indemnite_legale": round(indemnite_legale, 2),
            "indemnite_preavis": round(indemnite_preavis, 2),
            "total_minimum": round(indemnite_legale + indemnite_preavis, 2),
            "disclaimer": "Montants minimums légaux. Convention collective peut prévoir plus."
        }
    
    def traiter_demande(self, message_utilisateur: str) -> str:
        """Traite la demande de l'utilisateur avec l'agent IA"""
        
        self.historique_conversation.append({
            "role": "user",
            "content": message_utilisateur
        })
        
        response = self.client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=2000,
            system=self.construire_prompt_systeme(),
            messages=self.historique_conversation
        )
        
        reponse_agent = response.content[0].text
        
        self.historique_conversation.append({
            "role": "assistant",
            "content": reponse_agent
        })
        
        return reponse_agent

# Utilisation
agent = AgentJuridiqueIA(api_key="votre_cle_api")

# Simulation de conversation
reponse1 = agent.traiter_demande(
    "Mon employeur vient de me licencier après 5 ans. Ai-je droit à des indemnités ?"
)
print("Agent:", reponse1)

# Calcul direct d'indemnités
indemnites = agent.calculer_indemnites_licenciement(
    anciennete_annees=5.0,
    salaire_brut_mensuel=2800.0
)
print("\nCalcul automatique:", indemnites)

## Le Problème dans le Code Initial

Dans un agent conversationnel réel, pour gérer plusieurs échanges successifs, il faut maintenir en mémoire l'historique complet des messages dans une liste `messages`. Chaque appel API envoie cet historique complet, ce qui permet la continuité et la cohérence des échanges.

Un code minimaliste qui ne traite qu'un échange ne peut pas gérer de véritable dialogue contextualisé.


In [ ]:
from typing import Dict, List, Optional
import anthropic
import json

class AgentJuridiqueIA:
    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.historique_conversation = []
        self.contexte_utilisateur = {}
        
    def construire_prompt_systeme(self) -> str:
        return """Tu es un agent IA spécialisé en droit du travail français pour un cabinet d'avocats.

RÈGLES IMPÉRATIVES :
- Tu fournis uniquement des informations juridiques GÉNÉRALES
- Tu ne donnes JAMAIS de conseil juridique personnalisé
- Tu rappelles systématiquement que seul un avocat peut analyser une situation spécifique
- Tu inclus toujours un disclaimer sur la nature informative de tes réponses
- Tu recommandes la consultation d'un avocat pour tout cas concret

OUTILS DISPONIBLES :
Tu peux utiliser ces fonctions en les appelant dans ta réponse :

1. calculer_indemnites(anciennete_annees, salaire_brut_mensuel, type_licenciement)
   - Calcule les indemnités légales minimales
   - type_licenciement: "economique" ou "personnel"

2. proposer_rendez_vous(specialite)
   - Propose des créneaux avec un avocat
   - specialite: "droit_travail", "droit_commercial", etc.

Pour utiliser un outil, réponds avec ce format JSON :
{
  "action": "nom_fonction",
  "parametres": {...},
  "message_utilisateur": "ton message au client"
}

DÉMARCHE :
1. Accueillir et écouter la situation
2. Poser des questions de qualification si nécessaire
3. Fournir des informations générales
4. Calculer si demandé et données suffisantes
5. Recommander un avocat
6. Proposer un rendez-vous si le client est intéressé

Sois empathique, professionnel et précis."""

    def calculer_indemnites_licenciement(
        self, 
        anciennete_annees: float, 
        salaire_brut_mensuel: float,
        type_licenciement: str = "economique"
    ) -> Dict:
        """Calcul des indemnités légales minimales de licenciement"""
        
        # Indemnité légale de licenciement
        if anciennete_annees < 10:
            indemnite_legale = (anciennete_annees * 0.25) * salaire_brut_mensuel
        else:
            indemnite_legale = (10 * 0.25 + (anciennete_annees - 10) * 0.33) * salaire_brut_mensuel
        
        # Préavis (simplifié)
        if anciennete_annees < 2:
            preavis_mois = 1
        else:
            preavis_mois = 2
        
        indemnite_preavis = preavis_mois * salaire_brut_mensuel
        
        return {
            "anciennete": anciennete_annees,
            "salaire_brut": salaire_brut_mensuel,
            "type_licenciement": type_licenciement,
            "indemnite_legale": round(indemnite_legale, 2),
            "indemnite_preavis": round(indemnite_preavis, 2),
            "total_minimum": round(indemnite_legale + indemnite_preavis, 2)
        }
    
    def proposer_rendez_vous(self, specialite: str = "droit_travail") -> Dict:
        """Simule la proposition de créneaux de rendez-vous"""
        # En production, ceci interrogerait un vrai système de calendrier
        return {
            "specialite": specialite,
            "avocat": "Maître Sophie Dupont",
            "creneaux": [
                {"date": "2025-10-24", "heure": "14:00"},
                {"date": "2025-10-25", "heure": "10:30"},
                {"date": "2025-10-28", "heure": "16:00"}
            ],
            "duree": "45 minutes",
            "tarif": "150 € TTC"
        }
    
    def executer_action(self, action: str, parametres: Dict) -> Dict:
        """Exécute une action demandée par l'agent"""
        if action == "calculer_indemnites":
            return self.calculer_indemnites_licenciement(**parametres)
        elif action == "proposer_rendez_vous":
            return self.proposer_rendez_vous(**parametres)
        else:
            return {"erreur": f"Action inconnue: {action}"}
    
    def traiter_demande(self, message_utilisateur: str) -> tuple[str, Optional[Dict]]:
        """
        Traite la demande de l'utilisateur avec l'agent IA
        Retourne: (message_agent, resultat_action_si_executee)
        """
        
        # Ajouter le message utilisateur à l'historique
        self.historique_conversation.append({
            "role": "user",
            "content": message_utilisateur
        })
        
        # Appel au LLM
        response = self.client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=2000,
            system=self.construire_prompt_systeme(),
            messages=self.historique_conversation
        )
        
        reponse_agent = response.content[0].text
        
        # Vérifier si l'agent demande d'exécuter une action
        resultat_action = None
        try:
            # Tenter de parser si c'est un appel d'action JSON
            if "{" in reponse_agent and "action" in reponse_agent:
                # Extraire le JSON (simpliste, en prod utiliser un parser robuste)
                debut_json = reponse_agent.find("{")
                fin_json = reponse_agent.rfind("}") + 1
                json_str = reponse_agent[debut_json:fin_json]
                action_data = json.loads(json_str)
                
                if "action" in action_data:
                    # Exécuter l'action
                    resultat_action = self.executer_action(
                        action_data["action"],
                        action_data.get("parametres", {})
                    )
                    
                    # Construire la réponse avec le résultat
                    message_final = action_data.get("message_utilisateur", "")
                    
                    # Ajouter le résultat au contexte pour le prochain tour
                    self.historique_conversation.append({
                        "role": "assistant",
                        "content": f"{message_final}\n\nRésultat du calcul: {json.dumps(resultat_action, ensure_ascii=False)}"
                    })
                    
                    return message_final, resultat_action
        except:
            pass  # Si ce n'est pas un JSON valide, traiter comme réponse normale
        
        # Réponse normale sans action
        self.historique_conversation.append({
            "role": "assistant",
            "content": reponse_agent
        })
        
        return reponse_agent, resultat_action


# ============================================
# BOUCLE D'ÉCHANGE INTERACTIVE - VERSION CLI
# ============================================

def boucle_conversation_cli():
    """
    Boucle de conversation en ligne de commande
    C'est ICI que se passe la vraie interaction multi-tours
    """
    print("=" * 60)
    print("Agent IA Juridique - Cabinet d'Avocats")
    print("=" * 60)
    print("Tapez 'exit' ou 'quit' pour terminer la conversation\n")
    
    agent = AgentJuridiqueIA(api_key="votre_cle_api")
    
    # Message d'accueil initial
    print("🤖 Agent: Bonjour, je suis l'assistant virtuel du cabinet.")
    print("         Comment puis-je vous aider aujourd'hui ?\n")
    
    # BOUCLE INFINIE jusqu'à ce que l'utilisateur quitte
    while True:
        # Récupérer l'entrée utilisateur
        message_utilisateur = input("👤 Vous: ").strip()
        
        # Condition de sortie
        if message_utilisateur.lower() in ['exit', 'quit', 'sortir', 'au revoir']:
            print("\n🤖 Agent: Au revoir ! N'hésitez pas à nous recontacter.")
            break
        
        if not message_utilisateur:
            continue
        
        # Traiter le message avec l'agent
        print("\n🤖 Agent: ", end="", flush=True)
        reponse, action_resultat = agent.traiter_demande(message_utilisateur)
        
        # Afficher la réponse
        print(reponse)
        
        # Si une action a été exécutée, afficher le résultat formaté
        if action_resultat:
            print("\n📊 Résultat du calcul:")
            for cle, valeur in action_resultat.items():
                print(f"   • {cle}: {valeur}")
        
        print()  # Ligne vide pour la lisibilité


# ============================================
# BOUCLE D'ÉCHANGE - VERSION SCRIPT SIMULÉ
# ============================================

def simulation_conversation_complete():
    """
    Simule une conversation complète sans interaction CLI
    Utile pour tests et démonstrations
    """
    agent = AgentJuridiqueIA(api_key="votre_cle_api")
    
    # Liste des messages à échanger
    messages_utilisateur = [
        "Bonjour, mon employeur vient de me licencier après 5 ans d'ancienneté. Ai-je droit à des indemnités ?",
        "C'est un licenciement économique. Mon salaire brut est de 2 800 € par mois.",
        "Oui, je souhaite prendre rendez-vous.",
        "Le créneau du vendredi 25 octobre à 10h30 me convient parfaitement."
    ]
    
    print("=" * 70)
    print("SIMULATION DE CONVERSATION COMPLÈTE")
    print("=" * 70)
    print()
    
    # BOUCLE sur chaque message
    for i, message in enumerate(messages_utilisateur, 1):
        print(f"👤 Client (tour {i}): {message}")
        print()
        
        # L'agent traite le message
        reponse, action_resultat = agent.traiter_demande(message)
        
        print(f"🤖 Agent (tour {i}): {reponse}")
        
        if action_resultat:
            print("\n📊 Calcul effectué:")
            print(json.dumps(action_resultat, indent=2, ensure_ascii=False))
        
        print("\n" + "-" * 70 + "\n")
    
    print("✅ Conversation terminée avec succès")
    print(f"📝 Nombre total d'échanges dans l'historique: {len(agent.historique_conversation)}")


# ============================================
# BOUCLE D'ÉCHANGE - VERSION WEB/API
# ============================================

class AgentJuridiqueWeb:
    """
    Version adaptée pour une API web (FastAPI, Flask, etc.)
    Gère les sessions utilisateur avec stockage de l'historique
    """
    
    def __init__(self):
        self.sessions = {}  # Stockage des sessions {session_id: agent}
    
    def creer_session(self, session_id: str, api_key: str) -> str:
        """Crée une nouvelle session de conversation"""
        self.sessions[session_id] = AgentJuridiqueIA(api_key)
        return session_id
    
    def traiter_message(self, session_id: str, message: str) -> Dict:
        """
        Traite un message dans le contexte d'une session
        Retourne JSON pour l'API
        """
        if session_id not in self.sessions:
            return {
                "erreur": "Session inconnue",
                "code": 404
            }
        
        agent = self.sessions[session_id]
        reponse, action_resultat = agent.traiter_demande(message)
        
        return {
            "session_id": session_id,
            "reponse_agent": reponse,
            "action_executee": action_resultat,
            "nombre_echanges": len(agent.historique_conversation),
            "timestamp": "2025-10-22T11:30:00"
        }
    
    def terminer_session(self, session_id: str):
        """Termine et nettoie une session"""
        if session_id in self.sessions:
            del self.sessions[session_id]


# Exemple d'utilisation avec FastAPI (pseudo-code)
"""
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()
agent_web = AgentJuridiqueWeb()

class Message(BaseModel):
    session_id: str
    message: str

@app.post("/nouvelle-session")
def nouvelle_session():
    import uuid
    session_id = str(uuid.uuid4())
    agent_web.creer_session(session_id, "votre_cle_api")
    return {"session_id": session_id}

@app.post("/message")
def envoyer_message(msg: Message):
    # LA BOUCLE se passe côté client qui appelle cette API
    # à chaque nouveau message de l'utilisateur
    resultat = agent_web.traiter_message(msg.session_id, msg.message)
    return resultat

@app.post("/terminer-session")
def terminer_session(session_id: str):
    agent_web.terminer_session(session_id)
    return {"status": "session terminée"}
"""


# ============================================
# POINT D'ENTRÉE PRINCIPAL
# ============================================

if __name__ == "__main__":
    print("Choisissez le mode d'exécution:")
    print("1. Conversation interactive CLI")
    print("2. Simulation de conversation complète")
    
    choix = input("\nVotre choix (1 ou 2): ").strip()
    
    if choix == "1":
        boucle_conversation_cli()
    elif choix == "2":
        simulation_conversation_complete()
    else:
        print("Choix invalide")


## Le Mécanisme Clé : L’Historique de Conversation

**C’est l’historique qui permet la continuité**

**À chaque appel** :

- Message actuel + tous les messages précédents → LLM
- LLM voit **toute la conversation**
- LLM peut référencer des informations données 3 messages avant
- Réponse cohérente avec le contexte

In [ ]:
class AgentJuridiqueIA:
    def __init__(self):
        self.historique_conversation = []  # ← MÉMOIRE de la conversation
    
    def traiter_demande(self, message):
        # Ajouter le nouveau message
        self.historique_conversation.append({
            "role": "user",
            "content": message
        })
        
        # Envoyer TOUT l'historique au LLM
        response = self.client.messages.create(
            messages=self.historique_conversation  # ← Contexte complet
        )
        
        # Sauvegarder la réponse
        self.historique_conversation.append({
            "role": "assistant",
            "content": response.content[0].text
        })




## l’agent juridique n’est pas correctement codé
L’exemple d’agent juridique fourni auparavant **fonctionne logiquement**, mais il est **architecturalement incomplet** si on veut coller à la API moderne d’Anthropic (celle avec `tools=[]` plutôt que des ***appels simulés dans le prompt système*** ou du pseudo‑JSON).

### pourquoi l’ancien code n’est pas “correctement codé”


| Problème | Description | Conséquence |
| ----------- | ----------- | ----------- |
| Tools non déclarés dans tools=[]	| Tout est défini comme texte dans le prompt |	Pas de validation de schéma, risque de format incorrect |
|Parsing JSON manuel	|Le modèle génère du JSON approximatif|	Risque d’erreur de parsing, behavior fragile
|Role assistant mal utilisé	|Absence de tool_use / tool_result|	Ne suit pas le protocole API|
|Prompt trop verbeux	|Le système se charge de décrire tous les outils	|Gaspillage de tokens + incohérence si tool naming change|
|Aucune distinction entre “inference” et “post‑processing”|	Le modèle combine logique et exécution	|Peu clair et non optimisé pour automation|

### Ce qu’il faut faire


In [ ]:
response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    messages=[{"role": "user", "content": "Calcul des indemnités pour 5 ans et 2800 €."}],
    tools=[
        {
            "name": "calculer_indemnites_licenciement",
            "description": "Calcule les indemnités selon le Code du travail.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "anciennete_annees": {"type": "number"},
                    "salaire_brut_mensuel": {"type": "number"},
                    "type_licenciement": {"type": "string", "enum": ["economique", "personnel"]}
                },
                "required": ["anciennete_annees", "salaire_brut_mensuel", "type_licenciement"]
            }
        }
    ]
)

Et le modèle répond avec des content blocks structurés :

In [ ]:
[
  {"type": "text", "text": "Je vais calculer vos indemnités…"},
  {
    "type": "tool_use",
    "id": "toolu_123",
    "name": "calculer_indemnites_licenciement",
    "input": {
      "anciennete_annees": 5,
      "salaire_brut_mensuel": 2800,
      "type_licenciement": "economique"
    }
  }
]

le résultat du calcul en ajoutant dans messages :

In [ ]:
messages.append({
    "role": "user",
    "content": [
        {"type": "tool_result",
         "tool_use_id": "toolu_123",
         "content": json.dumps({"indemnite_legale": 3500, "total_minimum": 9100})}
    ]
})

## Version corrigée

In [ ]:
import anthropic
import json

# ======================================================
# CONFIGURATION DE L'AGENT
# ======================================================

client = anthropic.Anthropic(api_key="VOTRE_CLE_API")

# Définition des outils disponibles pour l'agent
tools = [
    {
        "name": "calculer_indemnites_licenciement",
        "description": (
            "Calcule les indemnités légales minimales de licenciement "
            "selon le Code du travail français."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "anciennete_annees": {
                    "type": "number",
                    "description": "Nombre d'années d'ancienneté du salarié."
                },
                "salaire_brut_mensuel": {
                    "type": "number",
                    "description": "Salaire brut mensuel en euros."
                },
                "type_licenciement": {
                    "type": "string",
                    "enum": ["economique", "personnel"],
                    "description": "Type de licenciement."
                }
            },
            "required": ["anciennete_annees", "salaire_brut_mensuel", "type_licenciement"]
        },
    }
]

# ======================================================
# IMPLÉMENTATION DE LA LOGIQUE LOCALE DU TOOL
# ======================================================

def calculer_indemnites_licenciement(anciennete_annees: float, salaire_brut_mensuel: float, type_licenciement: str):
    """
    Calcule les indemnités légales de licenciement
    - formule simplifiée :
      indemnité légale = 1/4 de mois par année d'ancienneté
      + préavis de 1 à 2 mois selon ancienneté
    """
    if anciennete_annees < 10:
        indemnite_legale = (anciennete_annees * 0.25) * salaire_brut_mensuel
    else:
        indemnite_legale = ((10 * 0.25) + ((anciennete_annees - 10) * 0.33)) * salaire_brut_mensuel

    preavis_mois = 1 if anciennete_annees < 2 else 2
    indemnite_preavis = salaire_brut_mensuel * preavis_mois

    total = indemnite_legale + indemnite_preavis

    return {
        "anciennete_annees": anciennete_annees,
        "salaire_brut": salaire_brut_mensuel,
        "type_licenciement": type_licenciement,
        "indemnite_legale": round(indemnite_legale, 2),
        "indemnite_preavis": round(indemnite_preavis, 2),
        "total_minimum": round(total, 2),
        "disclaimer": (
            "Montants estimatifs minimaux selon le Code du travail. "
            "La convention collective ou la situation individuelle peut prévoir davantage."
        )
    }

# ======================================================
# BOUCLE DE CONVERSATION SIMPLIFIÉE
# ======================================================

def conversation_agent_juridique():
    messages = [
        {
            "role": "system",
            "content": (
                "Tu es un agent conversationnel spécialisé en droit du travail français.\n"
                "Ton rôle est d'expliquer les principes généraux et de guider le client.\n"
                "Tu ne donnes pas de conseil juridique personnalisé.\n"
                "Si des informations sont suffisantes, appelle l’outil adéquat.\n"
                "Toujours rappeler que seul un avocat peut analyser un cas spécifique."
            )
        },
        {
            "role": "user",
            "content": "Bonjour, j’ai été licencié après 5 ans d’ancienneté avec un salaire brut de 2800 €."
        }
    ]

    while True:
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=1200,
            messages=messages,
            tools=tools,
            tool_choice={"type": "auto"}
        )

        # Vérifie si le modèle appelle un outil
        tool_calls = [block for block in response.content if block.type == "tool_use"]

        if not tool_calls:
            # Pas d'outil à exécuter : affichage direct de la réponse textuelle
            for block in response.content:
                if block.type == "text":
                    print(f"\n🤖 Agent: {block.text}")
            break

        # Si un outil est appelé
        for call in tool_calls:
            nom_fonction = call.name
            arguments = call.input
            print(f"\n🔧 Appel de tool: {nom_fonction} avec {arguments}")

            # Exécuter la fonction Python correspondante
            if nom_fonction == "calculer_indemnites_licenciement":
                result = calculer_indemnites_licenciement(**arguments)
            else:
                result = {"error": f"Tool non reconnu: {nom_fonction}"}

            print("📊 Résultat tool:", result)

            # Ajouter le résultat de l’outil à la conversation
            messages.append({
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": call.id,
                        "content": json.dumps(result, ensure_ascii=False)
                    }
                ]
            })

    print("\n--- Fin de la conversation ---")

# ======================================================
# POINT D’ENTRÉE PRINCIPAL
# ======================================================

if __name__ == "__main__":
    conversation_agent_juridique()